In [0]:
import os, sys
sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), "src"))
from lh import common

# silver/02_rebuild_silver: runs in the daily job only when the silver task failed.
# Masked tables refuse time travel and RESTORE, so Silver is rebuilt from Bronze instead:
# guard -> empty every Silver table (tables, masks and filters stay) -> reset watermarks -> replay -> check.
dbutils.widgets.dropdown("env", "dev", ["dev", "test", "prod"])
dbutils.widgets.dropdown("force", "no", ["no", "yes"])   # yes = rebuild without a failed checkpoint (dev tests only)
env, force = dbutils.widgets.get("env"), dbutils.widgets.get("force") == "yes"
cfg = common.load_config(env)
cat = cfg["catalog"]

last = spark.sql(f"""
    WITH c AS (SELECT * FROM {cat}.ops.dq_results WHERE rule = 'checkpoint')
    SELECT max(run_ts) >= current_timestamp() - INTERVAL 6 HOURS AS recent,
           sum(CASE WHEN status = 'fail' THEN 1 ELSE 0 END) AS failed
    FROM c WHERE run_ts = (SELECT max(run_ts) FROM c)""").first()
failed_now = bool(last.recent) and (last.failed or 0) > 0
if force and env != "dev":
    raise Exception("force is allowed in dev only")
if not (failed_now or force):
    raise Exception("refused: the latest Silver checkpoint did not fail in the last 6 hours, so there is nothing to rebuild")
print(f"{cat}: rebuilding Silver from Bronze", "(forced, dev test)" if force else f"({last.failed} tables failed the checkpoint)")

In [0]:
# Empty every Silver table and reset the watermarks. DELETE keeps each table, its column masks and row filters.
tables = [r.table_name for r in spark.sql(f"""SELECT table_name FROM {cat}.information_schema.tables
                                              WHERE table_schema = 'silver' AND table_type <> 'VIEW'
                                              ORDER BY table_name""").collect()]
for t in tables:
    spark.sql(f"DELETE FROM {cat}.silver.{t}")
spark.sql(f"DELETE FROM {cat}.ops.silver_state")
print(f"emptied {len(tables)} Silver tables and reset the watermarks:", ", ".join(tables))

In [0]:
# Replay: the normal Silver runner loads everything again from Bronze and runs the checkpoint
# (it raises if the checkpoint fails, so this task fails and Gold stays blocked).
import time
start = time.time()
dbutils.notebook.run("./01_load_silver", 3600, {"env": env})
check = spark.sql(f"""
    WITH c AS (SELECT * FROM {cat}.ops.dq_results WHERE rule = 'checkpoint')
    SELECT count(*) AS tables, sum(CASE WHEN status = 'fail' THEN 1 ELSE 0 END) AS failed
    FROM c WHERE run_ts = (SELECT max(run_ts) FROM c)""").first()
rows = sum(spark.table(f"{cat}.silver.{t}").count() for t in tables)
print(f"replay done in {time.time() - start:.0f}s: {rows} Silver rows, checkpoint {check.tables - check.failed}/{check.tables} pass")
if check.failed:
    common.log_run(spark, cfg, "silver_rebuild", "failed", rows, f"checkpoint still failing on {check.failed} tables")
    raise Exception("Silver rebuilt but the checkpoint still fails: Gold stays blocked")
common.log_run(spark, cfg, "silver_rebuild", "success", rows, f"{len(tables)} tables replayed from Bronze, checkpoint pass")